# Table Playground

Interactive exploration of tabular results — render as Markdown,
LaTeX (booktabs), or as a bar chart for visual comparison.

**Compatible experiments:** `fronthaul_table`
(uses `kind == 'table'`).

Note: `result.table_data` is a `Dict[algorithm_name, Dict[field, value]]`
structure — NOT a `SimResult`.  The `cordis.plotting.bar_chart` and
`to_*_table` functions expect a `SimResult`, so this notebook builds
tables and charts from `result.table_data` directly (matching what
`scripts/plot_fronthaul_table.py` does).

In [ ]:
# Stage 12: shared setup — make `cordis` importable when this notebook
# is launched from notebooks/, then apply the IEEE paper rcParams.
import sys, logging
from pathlib import Path
from _playground_helpers import (
    setup_paper_style, load_latest_result, load_run, summarize,
)

import numpy as np
import matplotlib.pyplot as plt

# Set use_latex=False if pdflatex isn't on PATH (e.g. on a compute node).
setup_paper_style(use_latex=True)

logging.basicConfig(level=logging.WARNING, format='%(levelname)-7s %(message)s')

In [ ]:
EXPERIMENT = 'fronthaul_table'

# Optional: load a specific run directory instead of the latest.
# Set to a path like 'results/exp_fronthaul_table/20260520_113500'
# (relative to the repo root) to re-render an older table.
EXP_DIR = None

result = load_result(EXPERIMENT, EXP_DIR)
assert result.kind == 'table'
summarize(result)
print()
print('Algorithms in table:', list(result.table_data.keys()))
first_algo = next(iter(result.table_data))
print(f'Fields per algorithm: {list(result.table_data[first_algo].keys())}')

## 1. Markdown render (good for previews + GitHub READMEs)

In [ ]:
# Mirrors what scripts/plot_fronthaul_table.py emits.
# real_scalars is PER COORDINATION ROUND (apples-to-apples across
# algorithms).  iterations holds the per-solve multiplier
# (1 for Centralized/Split, T_ADMM for ADMM).
lines = [
    '| Algorithm | Data shared | Size | Real scalars / round | Iterations | Total / solve | Scalable |',
    '|---|---|---|---|---|---|---|',
]
for algo, row in result.table_data.items():
    rs = row.get('real_scalars')
    it = row.get('iterations')
    rs_str = '—' if rs is None else (
        f'{rs:.1f}' if isinstance(rs, float) else f'{rs:d}'
    )
    it_str = '—' if it is None else (
        f'{it:.1f}' if isinstance(it, float) and not float(it).is_integer()
        else f'{int(it):d}'
    )
    tot_str = '—' if (rs is None or it is None) else (
        f'{float(rs) * float(it):.0f}'
    )
    lines.append(
        f"| {algo} | {row['data_to_share']} | {row['size']} | "
        f"{rs_str} | {it_str} | {tot_str} | "
        f"{'✓' if row['scalable'] else '✗'} |"
    )
print('\n'.join(lines))

## 2. LaTeX render for the paper

Uses `booktabs` style.  Copy-paste the output into your paper's `.tex`
(or use `\input{...}` from a saved `.tex` file).

In [ ]:
tex_lines = [
    r'\begin{table}[t]',
    r'\centering',
    r'\caption{Per-AP fronthaul coordination overhead.}',
    r'\label{tab:fronthaul}',
    r'\begin{tabular}{lcccc}',
    r'\toprule',
    r'Algorithm & Data shared & Size & Real / iter & Scalable \\',
    r'\midrule',
]
for algo, row in result.table_data.items():
    rs = row.get('real_scalars')
    rs_str = '--' if rs is None else (
        f'{rs:.1f}' if isinstance(rs, float) else f'{rs:d}'
    )
    tex_lines.append(
        f"{algo} & {row['data_to_share']} & {row['size']} & "
        f"{rs_str} & "
        + (r'$\checkmark$' if row['scalable'] else r'$\times$')
        + r' \\'
    )
tex_lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
tex = '\n'.join(tex_lines)
print(tex)

# Save it for \input{} from your paper.
out = Path('../figures/playground/fronthaul_table.tex')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(tex)
print(f'wrote {out}')

## 3. Bar chart — per-round fronthaul cost

All algorithms reported per coordination round (apples-to-apples).
Centralized + Split run 1 round per coherence block; CORDIS-ADMM
runs T_ADMM rounds — shown in the `iterations` column above.

When numbers span orders of magnitude (e.g. fronthaul scalars from
a few to a few hundred), a log-y bar chart often communicates the
gap better than the numeric table alone.  Built from
`result.table_data` directly with raw matplotlib (since
`cordis.plotting.bar_chart` expects a SimResult).

In [ ]:
from cordis.plotting import figsize

# Pull the per-round count for plotting, skip rows with None.
algs   = []
values = []
for algo, row in result.table_data.items():
    rs = row.get('real_scalars')
    if rs is None:
        continue
    algs.append(algo)
    values.append(float(rs))

fig, ax = plt.subplots(figsize=figsize(width='single', aspect=3/2))
bars = ax.bar(range(len(algs)), values, color='C0', edgecolor='black')
ax.set_yscale('log')
ax.set_xticks(range(len(algs)))
ax.set_xticklabels(algs, rotation=20, ha='right')
ax.set_ylabel('real scalars per AP per coordination round')
ax.set_title('Per-round fronthaul cost (log scale)')
ax.grid(True, axis='y', alpha=0.3)

# Annotate each bar with the count for clarity.
for rect, v in zip(bars, values):
    ax.annotate(f'{v:.0f}', xy=(rect.get_x() + rect.get_width()/2,
                                rect.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 4. Highlight CORDIS rows in the LaTeX output

For paper figures it's nice to bold-face the proposed algorithms
so the reader's eye lands on them.  This is a post-process on the
LaTeX text from cell 2.

In [ ]:
tex_with_highlight = tex
for alg in ('CORDIS-Split', 'CORDIS-ADMM'):
    tex_with_highlight = tex_with_highlight.replace(
        f'{alg}', f'\\textbf{{{alg}}}'
    )
print(tex_with_highlight)

## 5. Figsize variants

In [ ]:
# Figsize variants — `figsize` returns (w, h) in inches for matplotlib.
# width: 'single' (one column), 'double' (two-column), 'third' (3-up panel).
# aspect: w/h ratio.  Tweak both to fit your paper layout.
from cordis.plotting import figsize

for width in ('single', 'double', 'third'):
    w, h = figsize(width=width, aspect=3/2)
    print(f'{width:>6}: ({w:.2f}, {h:.2f}) inches')

# Example: tight three-up panel for a paper sub-figure
# fig, axes = plt.subplots(1, 3, figsize=figsize(width='double', aspect=3.5/1.5))

## 6. Save the bar chart with provenance

In [ ]:
# Save with provenance metadata (Git SHA, creation date, etc. — embedded
# into the PDF's metadata, prepended as comments in the .pgf).
from cordis.plotting import save_figure

# Adjust EXPERIMENT and metric labels to match the figure above.
out = save_figure(
    fig,
    base_path=f'../figures/playground/{EXPERIMENT}_demo',
    formats=('pdf', 'png'),       # add 'pgf' on systems with LaTeX
    metadata={'Experiment': EXPERIMENT, 'Notebook': 'playground'},
)
for p in out:
    print('wrote', p)